# NLP2 LoRA v2 Clean Training Experiment
# Goal:
# Test whether cleaner terminology-focused instruction-tuning data improves LoRA adaptation.
#
# RQ2 link: training-data modification through cleaner instruction-format examples.
# RQ3 link: model adaptation using LoRA.

In [5]:
!pip install -q transformers datasets accelerate peft trl sacrebleu sentencepiece protobuf
!pip uninstall -y torchao

In [6]:
import os
import re
import ast
import gc
import random
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
import sacrebleu

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [7]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
from google.colab import files

uploaded = files.upload()
print(uploaded.keys())

Saving nlp2_full_dataframe_all_experiments_500.csv to nlp2_full_dataframe_all_experiments_500 (1).csv
dict_keys(['nlp2_full_dataframe_all_experiments_500 (1).csv'])


In [9]:
# Change filename if the uploaded file has a different name
csv_name = list(uploaded.keys())[0]

df = pd.read_csv(csv_name)
print(df.shape)
print(df.columns)
df.head()

(500, 28)
Index(['source', 'reference', 'terms', 'base_basic', 'base_strong',
       'lora_no_term', 'lora_basic', 'lora_basic_cleaned', 'terms_dict',
       'qwen3b_basic', 'qwen3b_strong', 'qwen3b_no_term',
       'qwen3b_selected_source', 'qwen3b_selected',
       'qwen3b_selected_term_coverage', 'random_terms_dict', 'qwen3b_random',
       'selected_missing_terms', 'num_missing_selected',
       'qwen3b_selected_repaired', 'repaired_missing_terms',
       'num_missing_repaired', 'qwen3b_definition_prompt',
       'lora_cleaned_missing_terms', 'num_missing_lora_cleaned',
       'lora_basic_cleaned_repaired', 'lora_repaired_missing_terms',
       'num_missing_lora_repaired'],
      dtype='object')


,source,reference,terms,base_basic,base_strong,lora_no_term,lora_basic,lora_basic_cleaned,terms_dict,qwen3b_basic,...,num_missing_selected,qwen3b_selected_repaired,repaired_missing_terms,num_missing_repaired,qwen3b_definition_prompt,lora_cleaned_missing_terms,num_missing_lora_cleaned,lora_basic_cleaned_repaired,lora_repaired_missing_terms,num_missing_lora_repaired
0,This service describes the deployed (run-time)...,Dieser Service beschreibt den implementierten ...,"{'design': 'Entwurf', 'state': 'Zustand'}",Dieses Service beschreibt den ausgeführten (be...,Dieses Service beschreibt den ausgeführten (be...,Dieses Service beschreibt den ausgeführt- oder...,Dieses Service beschreibt den ausgeführt- oder...,Dieses Service beschreibt den ausgeführt- oder...,"{'design': 'Entwurf', 'state': 'Zustand'}",Dieser Service beschreibt den ausgeführten (la...,...,0,Dieser Service beschreibt den ausgeführten (La...,{},0,Dieser Service beschreibt den ausgeführten (La...,{},0,Dieses Service beschreibt den ausgeführt- oder...,{},0
1,Request a checkup to perform a health diagnost...,"Fordern Sie eine Kontrolle an, um eine Fehlerd...",{'check': 'Kontrolle'},"Kontrolliere bitte einen Überwachungsbesuch, u...",Kontrollieren Sie bitte einen Check auf die Du...,Gerüste ein Checkup um einen Gesundheitsdiagno...,Kontrollieren Sie Ihre Datenflussanalyse durch...,Kontrollieren Sie Ihre Datenflussanalyse durch...,{'check': 'Kontrolle'},"Eine Kontrolle anfordern, um eine gesundheitsd...",...,0,Kontrolle für eine Gesundheitsdiagnostik durch...,{},0,Eine Kontrolle für eine Gesundheitsdiagnostik ...,{},0,Kontrollieren Sie Ihre Datenflussanalyse durch...,{},0
2,The data product is still available in the pro...,Das Datenprodukt ist weiterhin in der Datenpro...,{'provider': 'Provider'},Der Datenprodukt ist noch im Angebot des Provi...,Der Datenprodukt ist immer noch im Liste der D...,Der Datenprodukt ist noch im Provider-Liste ve...,Der Datenprodukt ist noch im Provider-Liste ve...,Der Datenprodukt ist noch im Provider-Liste ve...,{'provider': 'Provider'},Der Datenprodukt ist im Datenproduktverzeichni...,...,0,Der Datenprodukt ist im Datenproduktverzeichni...,{},0,Das Datenprodukt ist im Provider-Datenproduktv...,{},0,Der Datenprodukt ist noch im Provider-Liste ve...,{},0
3,This availability match percentage is calculat...,Diese prozentuale Verfügbarkeitsübereinstimmun...,{'percentage': 'prozentual'},Diese Verfügbarkeits-Match-Prozentsatz wird au...,Diese Verfügbarkeitsübereinstimmung ist basier...,Diese Verfügbarkeits-Match-Prozentsatz wird au...,Dieses Verfügbarkeitsmatch-Prozentsatz wird au...,Dieses Verfügbarkeitsmatch-Prozentsatz wird au...,{'percentage': 'prozentual'},Diese prozentuale Verfügbarkeitsmatchquote wir...,...,0,Diese prozentuale Verfügbarkeitsmatchquote wir...,{},0,Diese prozentuale Verfügbarkeitsmatchquote wir...,{'percentage': 'prozentual'},1,Dieser Verfügbarkeitsmatch-prozentuale Wert wi...,{},0
4,Open the consumption model containing the meas...,Öffnen Sie das Verbrauchsmodell mit den Kennza...,{'consumption model': 'Verbrauchsmodell'},"Öffne das Verbrauchsmodell, das die Maßnahmen ...","Verwenden Sie den Verbrauchsmodell, der die Ma...",Öffnen Sie den Konsummodellinhalt mit den Maße...,Gerne! Hier ist die deutsche Übersetzung:\n\nÖ...,"Öffnen Sie das Verbrauchsmodell, das die Maße ...",{'consumption model': 'Verbrauchsmodell'},"Öffnen Sie das Verbrauchsmodell, das die Maßna...",...,0,"Verfügen Sie über das Verbrauchsmodell, das di...",{},0,"Öffne das Verbrauchsmodell, das die zu berücks...",{},0,"Öffnen Sie das Verbrauchsmodell, das die Maße ...",{},0


In [10]:
def parse_terms(x):
    if isinstance(x, dict):
        return x

    if pd.isna(x):
        return {}

    if isinstance(x, str):
        try:
            value = ast.literal_eval(x)
            if isinstance(value, dict):
                return value
        except Exception:
            pass

    return {}

df["terms_dict"] = df["terms"].apply(parse_terms)

print(df[["source", "reference", "terms", "terms_dict"]].head())
print("Rows:", len(df))

                                              source  \
0  This service describes the deployed (run-time)...   
1  Request a checkup to perform a health diagnost...   
2  The data product is still available in the pro...   
3  This availability match percentage is calculat...   
4  Open the consumption model containing the meas...   

                                           reference  \
0  Dieser Service beschreibt den implementierten ...   
1  Fordern Sie eine Kontrolle an, um eine Fehlerd...   
2  Das Datenprodukt ist weiterhin in der Datenpro...   
3  Diese prozentuale Verfügbarkeitsübereinstimmun...   
4  Öffnen Sie das Verbrauchsmodell mit den Kennza...   

                                       terms  \
0  {'design': 'Entwurf', 'state': 'Zustand'}   
1                     {'check': 'Kontrolle'}   
2                   {'provider': 'Provider'}   
3               {'percentage': 'prozentual'}   
4  {'consumption model': 'Verbrauchsmodell'}   

                                  ter

In [11]:
random.seed(42)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df = df.iloc[:400].copy()
eval_df = df.iloc[400:500].copy()

print("Train:", train_df.shape)
print("Eval:", eval_df.shape)

Train: (400, 28)
Eval: (100, 28)


In [12]:
def terms_to_lines(terms_dict):
    lines = []

    for source_term, target_term in terms_dict.items():
        lines.append(f"{source_term} -> {target_term}")

    return "\n".join(lines)


def make_basic_training_prompt(source, terms_dict):
    terms_text = terms_to_lines(terms_dict)

    prompt = f"""Translate the English sentence into German.
Use the provided terminology when relevant.
Return only the German translation.

English sentence:
{source}

Terminology:
{terms_text}

German translation:"""

    return prompt


def make_strong_training_prompt(source, terms_dict):
    terms_text = terms_to_lines(terms_dict)

    prompt = f"""Translate the English sentence into German.
You must use the exact German target term whenever the corresponding English source term appears.
Return only the German translation. Do not add explanations or introductions.

English sentence:
{source}

Terminology:
{terms_text}

German translation:"""

    return prompt

In [13]:
def build_lora_v2_training_rows(dataframe):
    rows = []

    for _, row in dataframe.iterrows():
        source = row["source"]
        reference = row["reference"]
        terms_dict = row["terms_dict"]

        basic_prompt = make_basic_training_prompt(source, terms_dict)
        strong_prompt = make_strong_training_prompt(source, terms_dict)

        rows.append({
            "prompt": basic_prompt,
            "completion": str(reference).strip(),
            "prompt_type": "basic"
        })

        rows.append({
            "prompt": strong_prompt,
            "completion": str(reference).strip(),
            "prompt_type": "strong"
        })

    return pd.DataFrame(rows)

train_sft_df = build_lora_v2_training_rows(train_df)

print(train_sft_df.shape)
train_sft_df.head()

(800, 3)


,prompt,completion,prompt_type
0,Translate the English sentence into German.\nU...,"Wählen Sie die Meldekategorie aus, die Sie erw...",basic
1,Translate the English sentence into German.\nY...,"Wählen Sie die Meldekategorie aus, die Sie erw...",strong
2,Translate the English sentence into German.\nU...,Version und überarbeitete Version,basic
3,Translate the English sentence into German.\nY...,Version und überarbeitete Version,strong
4,Translate the English sentence into German.\nU...,Folgende Historien- und Datenaustauschattribut...,basic


In [14]:
print("PROMPT:")
print(train_sft_df.iloc[0]["prompt"])
print("\nCOMPLETION:")
print(train_sft_df.iloc[0]["completion"])

PROMPT:
Translate the English sentence into German.
Use the provided terminology when relevant.
Return only the German translation.

English sentence:
Select the report category that you would like to extend.

Terminology:
category -> Kategorie

German translation:

COMPLETION:
Wählen Sie die Meldekategorie aus, die Sie erweitern möchten.


In [15]:
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Loaded:", base_model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct


In [16]:
def format_sft_example(example):
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["completion"]}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

train_dataset = Dataset.from_pandas(train_sft_df)
train_dataset = train_dataset.map(format_sft_example)

print(train_dataset[0]["text"][:1000])

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Translate the English sentence into German.
Use the provided terminology when relevant.
Return only the German translation.

English sentence:
Select the report category that you would like to extend.

Terminology:
category -> Kategorie

German translation:<|im_end|>
<|im_start|>assistant
Wählen Sie die Meldekategorie aus, die Sie erweitern möchten.<|im_end|>



In [17]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model_lora_v2 = get_peft_model(base_model, lora_config)
model_lora_v2.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [18]:
output_dir = "checkpoints/lora_v2_clean"

training_args = SFTConfig(
    output_dir=output_dir,
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model_lora_v2,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.353531
20,1.080331
30,0.934427
40,0.937147
50,0.890912


TrainOutput(global_step=50, training_loss=1.0392696952819824, metrics={'train_runtime': 139.4232, 'train_samples_per_second': 5.738, 'train_steps_per_second': 0.359, 'total_flos': 575175445204992.0, 'train_loss': 1.0392696952819824})

In [19]:
adapter_dir = "lora_v2_clean_adapter"

model_lora_v2.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("Saved adapter to:", adapter_dir)

Saved adapter to: lora_v2_clean_adapter


In [20]:
from google.colab import files

!zip -r lora_v2_clean_adapter.zip lora_v2_clean_adapter
files.download("lora_v2_clean_adapter.zip")

  adding: lora_v2_clean_adapter/ (stored 0%)
  adding: lora_v2_clean_adapter/adapter_model.safetensors (deflated 8%)
  adding: lora_v2_clean_adapter/adapter_config.json (deflated 58%)
  adding: lora_v2_clean_adapter/tokenizer_config.json (deflated 59%)
  adding: lora_v2_clean_adapter/tokenizer.json (deflated 81%)
  adding: lora_v2_clean_adapter/chat_template.jinja (deflated 71%)
  adding: lora_v2_clean_adapter/README.md (deflated 65%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
def make_no_term_prompt(source, terms_dict=None):
    prompt = f"""Translate the English sentence into German.
Return only the German translation. Do not add explanations or introductions.

English sentence:
{source}

German translation:"""
    return prompt


def make_basic_eval_prompt(source, terms_dict):
    terms_text = terms_to_lines(terms_dict)

    prompt = f"""Translate the English sentence into German.
Use the provided terminology when relevant.
Return only the German translation. Do not add explanations or introductions.

English sentence:
{source}

Terminology:
{terms_text}

German translation:"""
    return prompt


def make_strong_eval_prompt(source, terms_dict):
    terms_text = terms_to_lines(terms_dict)

    prompt = f"""Translate the English sentence into German.
You must use the exact German target term whenever the corresponding English source term appears.
Return only the German translation. Do not add explanations or introductions.

English sentence:
{source}

Terminology:
{terms_text}

German translation:"""
    return prompt

In [22]:
def generate_outputs(model, tokenizer, dataframe, prompt_function, batch_size=4, max_new_tokens=160):
    outputs = []

    model.eval()

    for start in range(0, len(dataframe), batch_size):
        batch = dataframe.iloc[start:start + batch_size]
        prompts = []

        for _, row in batch.iterrows():
            prompt = prompt_function(row["source"], row["terms_dict"])

            messages = [
                {"role": "user", "content": prompt}
            ]

            chat_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            prompts.append(chat_text)

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model.device)

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        for i in range(len(batch)):
            input_length = inputs["input_ids"][i].shape[0]
            new_tokens = generated[i][input_length:]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            outputs.append(text)

        print(f"Generated {len(outputs)} / {len(dataframe)}")

    return outputs

In [24]:
def compute_bleu(outputs, references):
    bleu = sacrebleu.corpus_bleu(
        outputs,
        [references]
    )
    return bleu.score


def compute_term_accuracy(outputs, terms_list):
    total_terms = 0
    matched_terms = 0

    for output, terms_dict in zip(outputs, terms_list):
        output_lower = str(output).lower()

        for source_term, target_term in terms_dict.items():
            total_terms += 1

            if str(target_term).lower() in output_lower:
                matched_terms += 1

    if total_terms == 0:
        return 0

    return matched_terms / total_terms


def has_boilerplate(text):
    text_lower = str(text).lower()

    patterns = [
        "gerne",
        "hier ist",
        "übersetzung",
        "deutsche übersetzung",
        "die deutsche übersetzung",
        "translation"
    ]

    for pattern in patterns:
        if pattern in text_lower:
            return True

    return False

In [25]:
eval_df = eval_df.copy()

eval_df["lora_v2_no_term"] = generate_outputs(
    model_lora_v2,
    tokenizer,
    eval_df,
    make_no_term_prompt,
    batch_size=4
)

eval_df["lora_v2_basic"] = generate_outputs(
    model_lora_v2,
    tokenizer,
    eval_df,
    make_basic_eval_prompt,
    batch_size=4
)

eval_df["lora_v2_strong"] = generate_outputs(
    model_lora_v2,
    tokenizer,
    eval_df,
    make_strong_eval_prompt,
    batch_size=4
)

eval_df[["source", "reference", "lora_v2_no_term", "lora_v2_basic", "lora_v2_strong"]].head()

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generated 4 / 100
Generated 8 / 100
Generated 12 / 100
Generated 16 / 100
Generated 20 / 100
Generated 24 / 100
Generated 28 / 100
Generated 32 / 100
Generated 36 / 100
Generated 40 / 100
Generated 44 / 100
Generated 48 / 100
Generated 52 / 100
Generated 56 / 100
Generated 60 / 100
Generated 64 / 100
Generated 68 / 100
Generated 72 / 100
Generated 76 / 100
Generated 80 / 100
Generated 84 / 100
Generated 88 / 100
Generated 92 / 100
Generated 96 / 100
Generated 100 / 100
Generated 4 / 100
Generated 8 / 100
Generated 12 / 100
Generated 16 / 100
Generated 20 / 100
Generated 24 / 100
Generated 28 / 100
Generated 32 / 100
Generated 36 / 100
Generated 40 / 100
Generated 44 / 100
Generated 48 / 100
Generated 52 / 100
Generated 56 / 100
Generated 60 / 100
Generated 64 / 100
Generated 68 / 100
Generated 72 / 100
Generated 76 / 100
Generated 80 / 100
Generated 84 / 100
Generated 88 / 100
Generated 92 / 100
Generated 96 / 100
Generated 100 / 100
Generated 4 / 100
Generated 8 / 100
Generated 12 / 1

,source,reference,lora_v2_no_term,lora_v2_basic,lora_v2_strong
400,Sharing a Schedule,Einen Terminplan teilen,Teilen Sie eine Sichtungsplanung,Teilen eines Terminplans,Teilen eines Terminplans
401,Expired Service Contracts (Net Value of),Ausgelaufene Serviceverträge (Nettowert),Verfallende Dienstleistungskontrakte (Netzmitt...,Verfallsverträge (Netzwert),Verfallsverträge (Netzwert)
402,Create a New Role Collection.,Legen Sie eine New Role Collection an.,Erstellen Sie eine neue Rolle Sammlung.,Erstellen Sie eine neue Rolle-Collection.,Erstellen Sie eine neue Rolle-Collection.
403,Note for SAP Cloud Platform customers,Anmerkung für Kunden von SAP Cloud Platform,Hinweis für Kunden von SAP Cloud Platform,Anmerkung für SAP Cloud Platform Kunden,Anmerkung für SAP Cloud Platform Kunden
404,Deletes a period.,Löscht einen Zeitraum.,Löscht einen Punkt.,Löscht ein Zeitraum.,Löscht ein Zeitraum.


In [26]:
gc.collect()
torch.cuda.empty_cache()

base_eval_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

eval_df["base_eval_no_term"] = generate_outputs(
    base_eval_model,
    tokenizer,
    eval_df,
    make_no_term_prompt,
    batch_size=4
)

eval_df["base_eval_basic"] = generate_outputs(
    base_eval_model,
    tokenizer,
    eval_df,
    make_basic_eval_prompt,
    batch_size=4
)

eval_df["base_eval_strong"] = generate_outputs(
    base_eval_model,
    tokenizer,
    eval_df,
    make_strong_eval_prompt,
    batch_size=4
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Generated 4 / 100
Generated 8 / 100
Generated 12 / 100
Generated 16 / 100
Generated 20 / 100
Generated 24 / 100
Generated 28 / 100
Generated 32 / 100
Generated 36 / 100
Generated 40 / 100
Generated 44 / 100
Generated 48 / 100
Generated 52 / 100
Generated 56 / 100
Generated 60 / 100
Generated 64 / 100
Generated 68 / 100
Generated 72 / 100
Generated 76 / 100
Generated 80 / 100
Generated 84 / 100
Generated 88 / 100
Generated 92 / 100
Generated 96 / 100
Generated 100 / 100
Generated 4 / 100
Generated 8 / 100
Generated 12 / 100
Generated 16 / 100
Generated 20 / 100
Generated 24 / 100
Generated 28 / 100
Generated 32 / 100
Generated 36 / 100
Generated 40 / 100
Generated 44 / 100
Generated 48 / 100
Generated 52 / 100
Generated 56 / 100
Generated 60 / 100
Generated 64 / 100
Generated 68 / 100
Generated 72 / 100
Generated 76 / 100
Generated 80 / 100
Generated 84 / 100
Generated 88 / 100
Generated 92 / 100
Generated 96 / 100
Generated 100 / 100
Generated 4 / 100
Generated 8 / 100
Generated 12 / 1

In [27]:
lora_v2_rows = []

methods = [
    "base_eval_no_term",
    "base_eval_basic",
    "base_eval_strong",
    "lora_v2_no_term",
    "lora_v2_basic",
    "lora_v2_strong"
]

for method in methods:
    bleu = compute_bleu(
        eval_df[method].tolist(),
        eval_df["reference"].tolist()
    )

    term_acc = compute_term_accuracy(
        eval_df[method].tolist(),
        eval_df["terms_dict"].tolist()
    )

    boilerplate_count = eval_df[method].apply(has_boilerplate).sum()

    lora_v2_rows.append({
        "method": method,
        "BLEU": bleu,
        "term_accuracy": term_acc,
        "boilerplate_count": boilerplate_count
    })

lora_v2_results_df = pd.DataFrame(lora_v2_rows)
lora_v2_results_df

,method,BLEU,term_accuracy,boilerplate_count
0,base_eval_no_term,19.093238,0.300000,0
1,base_eval_basic,17.559673,0.709091,0
2,base_eval_strong,15.330835,0.663636,0
3,lora_v2_no_term,20.981343,0.300000,1
4,lora_v2_basic,23.236286,0.772727,0
5,lora_v2_strong,21.380252,0.772727,0


In [28]:
eval_df.to_csv("nlp2_lora_v2_clean_eval_outputs_100.csv", index=False)
lora_v2_results_df.to_csv("nlp2_lora_v2_clean_eval_metrics_100.csv", index=False)

!zip nlp2_lora_v2_clean_experiment_100.zip \
nlp2_lora_v2_clean_eval_outputs_100.csv \
nlp2_lora_v2_clean_eval_metrics_100.csv \
lora_v2_clean_adapter.zip

files.download("nlp2_lora_v2_clean_experiment_100.zip")

  adding: nlp2_lora_v2_clean_eval_outputs_100.csv (deflated 83%)
  adding: nlp2_lora_v2_clean_eval_metrics_100.csv (deflated 44%)
  adding: lora_v2_clean_adapter.zip (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
full500_df = df.copy()

print(full500_df.shape)
print(full500_df.columns)

(500, 28)
Index(['source', 'reference', 'terms', 'base_basic', 'base_strong',
       'lora_no_term', 'lora_basic', 'lora_basic_cleaned', 'terms_dict',
       'qwen3b_basic', 'qwen3b_strong', 'qwen3b_no_term',
       'qwen3b_selected_source', 'qwen3b_selected',
       'qwen3b_selected_term_coverage', 'random_terms_dict', 'qwen3b_random',
       'selected_missing_terms', 'num_missing_selected',
       'qwen3b_selected_repaired', 'repaired_missing_terms',
       'num_missing_repaired', 'qwen3b_definition_prompt',
       'lora_cleaned_missing_terms', 'num_missing_lora_cleaned',
       'lora_basic_cleaned_repaired', 'lora_repaired_missing_terms',
       'num_missing_lora_repaired'],
      dtype='object')


In [30]:
full500_df["lora_v2_500_no_term"] = generate_outputs(
    model_lora_v2,
    tokenizer,
    full500_df,
    make_no_term_prompt,
    batch_size=4
)

full500_df["lora_v2_500_basic"] = generate_outputs(
    model_lora_v2,
    tokenizer,
    full500_df,
    make_basic_eval_prompt,
    batch_size=4
)

full500_df["lora_v2_500_strong"] = generate_outputs(
    model_lora_v2,
    tokenizer,
    full500_df,
    make_strong_eval_prompt,
    batch_size=4
)

full500_df[["source", "reference", "lora_v2_500_no_term", "lora_v2_500_basic", "lora_v2_500_strong"]].head()

Generated 4 / 500
Generated 8 / 500
Generated 12 / 500
Generated 16 / 500
Generated 20 / 500
Generated 24 / 500
Generated 28 / 500
Generated 32 / 500
Generated 36 / 500
Generated 40 / 500
Generated 44 / 500
Generated 48 / 500
Generated 52 / 500
Generated 56 / 500
Generated 60 / 500
Generated 64 / 500
Generated 68 / 500
Generated 72 / 500
Generated 76 / 500
Generated 80 / 500
Generated 84 / 500
Generated 88 / 500
Generated 92 / 500
Generated 96 / 500
Generated 100 / 500
Generated 104 / 500
Generated 108 / 500
Generated 112 / 500
Generated 116 / 500
Generated 120 / 500
Generated 124 / 500
Generated 128 / 500
Generated 132 / 500
Generated 136 / 500
Generated 140 / 500
Generated 144 / 500
Generated 148 / 500
Generated 152 / 500
Generated 156 / 500
Generated 160 / 500
Generated 164 / 500
Generated 168 / 500
Generated 172 / 500
Generated 176 / 500
Generated 180 / 500
Generated 184 / 500
Generated 188 / 500
Generated 192 / 500
Generated 196 / 500
Generated 200 / 500
Generated 204 / 500
Genera

,source,reference,lora_v2_500_no_term,lora_v2_500_basic,lora_v2_500_strong
0,Select the report category that you would like...,"Wählen Sie die Meldekategorie aus, die Sie erw...","Wählen Sie die Report-Kategorie aus, die erwei...","Wählen Sie die Kategorie aus, die erweitert we...","Wählen Sie die Kategorie aus, die erweitert we..."
1,Version and Revision,Version und überarbeitete Version,Version und Revision,Version und Überarbeitete Version,Version und Überarbeitete Version
2,The following history and data sharing attribu...,Folgende Historien- und Datenaustauschattribut...,Die folgenden historische und Datenverteilungs...,Die folgenden Historie- und Datenverteilungsei...,Die folgenden Historie- und Datenverteilungsat...
3,Enable access to SAP HANA monitoring views in ...,Aktivieren Sie den Zugriff auf SAP-HANA-Überwa...,Öffnen Sie Zugriff auf SAP HANA-Monitoringansi...,Öffnen Sie Zugriff auf die Überwachungsansicht...,Öffnen Sie Zugriff auf SAP HANA-Überwachungsan...
4,Only these users will be able to see and work ...,Nur diese Benutzer können Ihren Space sehen un...,Nur diese Benutzer können in Ihrem Raum sehen ...,Nur diese Benutzer können in Ihrem Raum sehen ...,Nur diese Benutzer können in Ihrem Raum sehen ...


In [31]:
lora_v2_500_rows = []

methods = [
    "lora_v2_500_no_term",
    "lora_v2_500_basic",
    "lora_v2_500_strong"
]

for method in methods:
    bleu = compute_bleu(
        full500_df[method].tolist(),
        full500_df["reference"].tolist()
    )

    term_acc = compute_term_accuracy(
        full500_df[method].tolist(),
        full500_df["terms_dict"].tolist()
    )

    boilerplate_count = full500_df[method].apply(has_boilerplate).sum()

    lora_v2_500_rows.append({
        "method": method,
        "BLEU": bleu,
        "term_accuracy": term_acc,
        "boilerplate_count": boilerplate_count
    })

lora_v2_500_results_df = pd.DataFrame(lora_v2_500_rows)
lora_v2_500_results_df

,method,BLEU,term_accuracy,boilerplate_count
0,lora_v2_500_no_term,22.106334,0.285458,2
1,lora_v2_500_basic,25.640417,0.700180,1
2,lora_v2_500_strong,24.890535,0.712747,1


In [32]:
lora_v2_500_comparison_df = pd.DataFrame([
    {
        "method": "Old LoRA basic",
        "BLEU": 12.455549,
        "term_accuracy": 0.628366,
        "boilerplate_count": 157,
        "note": "original 500 run"
    },
    {
        "method": "Old LoRA basic + cleanup",
        "BLEU": 14.168598,
        "term_accuracy": 0.626571,
        "boilerplate_count": 83,
        "note": "original 500 run"
    },
    {
        "method": "Old LoRA cleaned + repair",
        "BLEU": 19.268173,
        "term_accuracy": 0.928187,
        "boilerplate_count": 36,
        "note": "repair diagnostic"
    },
    {
        "method": "LoRA v2 no-term",
        "BLEU": lora_v2_500_results_df.loc[lora_v2_500_results_df["method"] == "lora_v2_500_no_term", "BLEU"].iloc[0],
        "term_accuracy": lora_v2_500_results_df.loc[lora_v2_500_results_df["method"] == "lora_v2_500_no_term", "term_accuracy"].iloc[0],
        "boilerplate_count": lora_v2_500_results_df.loc[lora_v2_500_results_df["method"] == "lora_v2_500_no_term", "boilerplate_count"].iloc[0],
        "note": "mixed train/eval 500 diagnostic"
    },
    {
        "method": "LoRA v2 basic",
        "BLEU": lora_v2_500_results_df.loc[lora_v2_500_results_df["method"] == "lora_v2_500_basic", "BLEU"].iloc[0],
        "term_accuracy": lora_v2_500_results_df.loc[lora_v2_500_results_df["method"] == "lora_v2_500_basic", "term_accuracy"].iloc[0],
        "boilerplate_count": lora_v2_500_results_df.loc[lora_v2_500_results_df["method"] == "lora_v2_500_basic", "boilerplate_count"].iloc[0],
        "note": "mixed train/eval 500 diagnostic"
    },
    {
        "method": "LoRA v2 strong",
        "BLEU": lora_v2_500_results_df.loc[lora_v2_500_results_df["method"] == "lora_v2_500_strong", "BLEU"].iloc[0],
        "term_accuracy": lora_v2_500_results_df.loc[lora_v2_500_results_df["method"] == "lora_v2_500_strong", "term_accuracy"].iloc[0],
        "boilerplate_count": lora_v2_500_results_df.loc[lora_v2_500_results_df["method"] == "lora_v2_500_strong", "boilerplate_count"].iloc[0],
        "note": "mixed train/eval 500 diagnostic"
    }
])

lora_v2_500_comparison_df

,method,BLEU,term_accuracy,boilerplate_count,note
0,Old LoRA basic,12.455549,0.628366,157,original 500 run
1,Old LoRA basic + cleanup,14.168598,0.626571,83,original 500 run
2,Old LoRA cleaned + repair,19.268173,0.928187,36,repair diagnostic
3,LoRA v2 no-term,22.106334,0.285458,2,mixed train/eval 500 diagnostic
4,LoRA v2 basic,25.640417,0.700180,1,mixed train/eval 500 diagnostic
5,LoRA v2 strong,24.890535,0.712747,1,mixed train/eval 500 diagnostic


In [33]:
full500_df.to_csv("nlp2_lora_v2_clean_outputs_500_mixed.csv", index=False)
lora_v2_500_results_df.to_csv("nlp2_lora_v2_clean_metrics_500_mixed.csv", index=False)
lora_v2_500_comparison_df.to_csv("nlp2_lora_v2_clean_comparison_500_mixed.csv", index=False)

!zip nlp2_lora_v2_clean_500_mixed_experiment.zip \
nlp2_lora_v2_clean_outputs_500_mixed.csv \
nlp2_lora_v2_clean_metrics_500_mixed.csv \
nlp2_lora_v2_clean_comparison_500_mixed.csv \
lora_v2_clean_adapter.zip

from google.colab import files
files.download("nlp2_lora_v2_clean_500_mixed_experiment.zip")

  adding: nlp2_lora_v2_clean_outputs_500_mixed.csv (deflated 83%)
  adding: nlp2_lora_v2_clean_metrics_500_mixed.csv (deflated 29%)
  adding: nlp2_lora_v2_clean_comparison_500_mixed.csv (deflated 45%)
  adding: lora_v2_clean_adapter.zip (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
def get_missing_terms(output, terms_dict):
    output_lower = str(output).lower()
    missing = {}

    for source_term, target_term in terms_dict.items():
        if str(target_term).lower() not in output_lower:
            missing[source_term] = target_term

    return missing


eval_df["lora_v2_basic_missing_terms"] = eval_df.apply(
    lambda row: get_missing_terms(row["lora_v2_basic"], row["terms_dict"]),
    axis=1
)

eval_df["num_missing_lora_v2_basic"] = eval_df["lora_v2_basic_missing_terms"].apply(len)

print(eval_df["num_missing_lora_v2_basic"].value_counts().sort_index())
print("Rows needing repair:", (eval_df["num_missing_lora_v2_basic"] > 0).sum())

eval_df[eval_df["num_missing_lora_v2_basic"] > 0][
    ["source", "terms_dict", "lora_v2_basic", "lora_v2_basic_missing_terms"]
].head()

num_missing_lora_v2_basic
0    78
1    21
4     1
Name: count, dtype: int64
Rows needing repair: 22


,source,terms_dict,lora_v2_basic,lora_v2_basic_missing_terms
401,Expired Service Contracts (Net Value of),{'expired': 'ausgelaufen'},Verfallsverträge (Netzwert),{'expired': 'ausgelaufen'}
407,Enter Query Name:,{'query': 'Query'},Geben Sie den Namen der Anfrage ein:,{'query': 'Query'}
409,Check the output panel.,{'panel': 'Bereich'},Überprüfen Sie den Ausgabepaneldienst.,{'panel': 'Bereich'}
414,To use new functionality of an adapter or upda...,{'use': 'Nutzung'},Um neue Funktionalität eines Adapters oder die...,{'use': 'Nutzung'}
416,Space Management Redesign,{'space': 'Space'},Ressourcemanagement-Redesign,{'space': 'Space'}


In [35]:
def make_lora_v2_repair_prompt(source, current_translation, missing_terms):
    term_lines = []

    for source_term, target_term in missing_terms.items():
        term_lines.append(f"{source_term} -> {target_term}")

    missing_text = "\n".join(term_lines)

    prompt = f"""You are revising a German machine translation.

The current translation is mostly correct, but it is missing required terminology.
Revise the translation so that it includes the required German terms.
Keep the meaning of the original English sentence.
Return only the revised German translation.
Do not add explanations or introductions.

English sentence:
{source}

Current German translation:
{current_translation}

Required missing terminology:
{missing_text}

Revised German translation:"""

    return prompt

In [37]:
def generate_lora_v2_repairs(dataframe, batch_size=4, max_new_tokens=160):
    repaired_outputs = dataframe["lora_v2_basic"].tolist()

    repair_indices = dataframe.index[dataframe["num_missing_lora_v2_basic"] > 0].tolist()
    print("Repairing", len(repair_indices), "LoRA v2 outputs")

    # Map dataframe index labels to Python list positions
    index_to_position = {}
    for position, index_label in enumerate(dataframe.index):
        index_to_position[index_label] = position

    model_lora_v2.eval()

    for start in range(0, len(repair_indices), batch_size):
        batch_indices = repair_indices[start:start + batch_size]
        prompts = []

        for idx in batch_indices:
            row = dataframe.loc[idx]

            prompt = make_lora_v2_repair_prompt(
                row["source"],
                row["lora_v2_basic"],
                row["lora_v2_basic_missing_terms"]
            )

            messages = [
                {"role": "user", "content": prompt}
            ]

            chat_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            prompts.append(chat_text)

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model_lora_v2.device)

        with torch.no_grad():
            generated = model_lora_v2.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        for i, idx in enumerate(batch_indices):
            input_length = inputs["input_ids"][i].shape[0]
            new_tokens = generated[i][input_length:]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

            position = index_to_position[idx]
            repaired_outputs[position] = text

        print(f"Repaired {min(start + batch_size, len(repair_indices))} / {len(repair_indices)}")

    return repaired_outputs

In [38]:
eval_df["lora_v2_basic_repaired"] = generate_lora_v2_repairs(eval_df, batch_size=4)

eval_df[eval_df["num_missing_lora_v2_basic"] > 0][
    ["source", "terms_dict", "lora_v2_basic", "lora_v2_basic_repaired"]
].head()

Repairing 22 LoRA v2 outputs
Repaired 4 / 22
Repaired 8 / 22
Repaired 12 / 22
Repaired 16 / 22
Repaired 20 / 22
Repaired 22 / 22


,source,terms_dict,lora_v2_basic,lora_v2_basic_repaired
401,Expired Service Contracts (Net Value of),{'expired': 'ausgelaufen'},Verfallsverträge (Netzwert),Ablaufverträge (Netzwert)
407,Enter Query Name:,{'query': 'Query'},Geben Sie den Namen der Anfrage ein:,Einen Namen für die Anfrage eingeben:
409,Check the output panel.,{'panel': 'Bereich'},Überprüfen Sie den Ausgabepaneldienst.,Prüfen Sie den Bereich des Ausgabepanels.
414,To use new functionality of an adapter or upda...,{'use': 'Nutzung'},Um neue Funktionalität eines Adapters oder die...,Nutzung von neuen Funktionen eines Adapters od...
416,Space Management Redesign,{'space': 'Space'},Ressourcemanagement-Redesign,Raumverwaltung-Redesign


In [39]:
lora_v2_repair_rows = []

methods = [
    "base_eval_basic",
    "lora_v2_basic",
    "lora_v2_strong",
    "lora_v2_basic_repaired"
]

for method in methods:
    bleu = compute_bleu(
        eval_df[method].tolist(),
        eval_df["reference"].tolist()
    )

    term_acc = compute_term_accuracy(
        eval_df[method].tolist(),
        eval_df["terms_dict"].tolist()
    )

    boilerplate_count = eval_df[method].apply(has_boilerplate).sum()

    lora_v2_repair_rows.append({
        "method": method,
        "BLEU": bleu,
        "term_accuracy": term_acc,
        "boilerplate_count": boilerplate_count
    })

lora_v2_repair_df = pd.DataFrame(lora_v2_repair_rows)
lora_v2_repair_df

,method,BLEU,term_accuracy,boilerplate_count
0,base_eval_basic,17.559673,0.709091,0
1,lora_v2_basic,23.236286,0.772727,0
2,lora_v2_strong,21.380252,0.772727,0
3,lora_v2_basic_repaired,24.139542,0.881818,0


In [40]:
eval_df["lora_v2_repaired_missing_terms"] = eval_df.apply(
    lambda row: get_missing_terms(row["lora_v2_basic_repaired"], row["terms_dict"]),
    axis=1
)

eval_df["num_missing_lora_v2_repaired"] = eval_df["lora_v2_repaired_missing_terms"].apply(len)

print("Before LoRA v2 repair:")
print(eval_df["num_missing_lora_v2_basic"].value_counts().sort_index())

print("\nAfter LoRA v2 repair:")
print(eval_df["num_missing_lora_v2_repaired"].value_counts().sort_index())

Before LoRA v2 repair:
num_missing_lora_v2_basic
0    78
1    21
4     1
Name: count, dtype: int64

After LoRA v2 repair:
num_missing_lora_v2_repaired
0    88
1    11
2     1
Name: count, dtype: int64


In [41]:
eval_df.to_csv("nlp2_lora_v2_repair_eval_outputs_100.csv", index=False)
lora_v2_repair_df.to_csv("nlp2_lora_v2_repair_eval_metrics_100.csv", index=False)

lora_v2_repair_missing_counts_df = pd.DataFrame({
    "stage": ["before_lora_v2_repair", "after_lora_v2_repair"],
    "fully_satisfied_outputs": [
        (eval_df["num_missing_lora_v2_basic"] == 0).sum(),
        (eval_df["num_missing_lora_v2_repaired"] == 0).sum()
    ],
    "outputs_with_missing_terms": [
        (eval_df["num_missing_lora_v2_basic"] > 0).sum(),
        (eval_df["num_missing_lora_v2_repaired"] > 0).sum()
    ]
})

lora_v2_repair_missing_counts_df.to_csv(
    "nlp2_lora_v2_repair_missing_counts_100.csv",
    index=False
)

!zip nlp2_lora_v2_repair_experiment_100.zip \
nlp2_lora_v2_repair_eval_outputs_100.csv \
nlp2_lora_v2_repair_eval_metrics_100.csv \
nlp2_lora_v2_repair_missing_counts_100.csv \
lora_v2_clean_adapter.zip

from google.colab import files
files.download("nlp2_lora_v2_repair_experiment_100.zip")

  adding: nlp2_lora_v2_repair_eval_outputs_100.csv (deflated 84%)
  adding: nlp2_lora_v2_repair_eval_metrics_100.csv (deflated 40%)
  adding: nlp2_lora_v2_repair_missing_counts_100.csv (deflated 24%)
  adding: lora_v2_clean_adapter.zip (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [42]:
eval_df.to_csv("nlp2_lora_v2_repair_eval_outputs_100.csv", index=False)
lora_v2_repair_df.to_csv("nlp2_lora_v2_repair_eval_metrics_100.csv", index=False)

lora_v2_repair_missing_counts_df = pd.DataFrame({
    "stage": ["before_lora_v2_repair", "after_lora_v2_repair"],
    "fully_satisfied_outputs": [
        (eval_df["num_missing_lora_v2_basic"] == 0).sum(),
        (eval_df["num_missing_lora_v2_repaired"] == 0).sum()
    ],
    "outputs_with_missing_terms": [
        (eval_df["num_missing_lora_v2_basic"] > 0).sum(),
        (eval_df["num_missing_lora_v2_repaired"] > 0).sum()
    ]
})

lora_v2_repair_missing_counts_df.to_csv(
    "nlp2_lora_v2_repair_missing_counts_100.csv",
    index=False
)

!zip nlp2_lora_v2_repair_experiment_100.zip \
nlp2_lora_v2_repair_eval_outputs_100.csv \
nlp2_lora_v2_repair_eval_metrics_100.csv \
nlp2_lora_v2_repair_missing_counts_100.csv \
lora_v2_clean_adapter.zip

from google.colab import files
files.download("nlp2_lora_v2_repair_experiment_100.zip")

updating: nlp2_lora_v2_repair_eval_outputs_100.csv (deflated 84%)
updating: nlp2_lora_v2_repair_eval_metrics_100.csv (deflated 40%)
updating: nlp2_lora_v2_repair_missing_counts_100.csv (deflated 24%)
updating: lora_v2_clean_adapter.zip (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>